# Landsat 8 LST 지도 시작 코드

이 노트북은 실제 GEE 자료를 조회하는 별도 실습입니다. **예시 웹보고서의 합성 그림을 생성하는 코드가 아닙니다. 이 저장소 작성 시 EE 인증·실행은 검증하지 않았습니다.** 본인의 승인된 EE Cloud 프로젝트 ID를 입력하세요. 계정 키나 인증 토큰은 저장하지 않습니다.

In [ ]:
%pip -q install earthengine-api geemap
import ee, geemap
PROJECT_ID = 'YOUR_EE_PROJECT_ID'
if PROJECT_ID == 'YOUR_EE_PROJECT_ID':
    raise ValueError('본인의 EE 프로젝트 ID를 먼저 입력하세요.')
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
# 부산 주변의 예시 사각형. 행정구역 경계가 아니다.
roi = ee.Geometry.Rectangle([129.0,35.08,129.18,35.25])
images = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(roi).filterDate('2020-06-01','2020-09-01')
          .filter(ee.Filter.eq('PROCESSING_LEVEL','L2SP')))
if images.size().getInfo() == 0:
    raise ValueError('조건을 만족하는 영상이 없습니다.')
# 한 장면만 표시한다. 계절 평균이 아니다.
image = ee.Image(images.sort('CLOUD_COVER').first())
qa = image.select('QA_PIXEL')
good = qa.bitwiseAnd(63).eq(0).And(image.select('QA_RADSAT').eq(0))
lst = (image.select('ST_B10').multiply(0.00341802).add(149.0)
       .subtract(273.15).updateMask(good).rename('LST'))
m = geemap.Map()
m.centerObject(roi,11)
m.addLayer(lst.clip(roi),{'min':15,'max':45,'palette':['2166ac','ffffbf','b2182b']},'한 장면 LST (°C)')
print('장면 날짜:',image.date().format('YYYY-MM-dd').getInfo())
display(m)

## 다음 분석은 조별로 설계합니다
지역·기간·수면 처리·유효 관측 기준·NDVI 산출·표본 단위·집계 방식을 정하세요. 이 시작 코드는 LST 지도 표시까지이며, 실제 녹지-LST 연구 전체를 완성한 코드가 아닙니다.